In [3]:
from collections import Counter, defaultdict
import csv


In [1]:
sentences = []

with open("train.src.tok", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        # if i >= 100000:
        #     break
        words = line.strip().split()
        if words:
            sentences.append(words)


In [ ]:
import pandas as pd

with open("train.src.tok", encoding="utf-8") as f:
    df = pd.DataFrame({"sentence": [line.rstrip("\n") for line in f]})

df.to_parquet("train.parquet", index=False)

In [ ]:
new_df = pd.read_parquet("train.parquet")

new_df.head()

In [ ]:
import polars as pl

In [ ]:
polars_df = pl.read_parquet("train.parquet")

polars_df.head()

In [ ]:
import nltk
from nltk.util import trigrams

nltk.download("punkt")

trigram_list = []
for sentence in polars_df["sentence"]:
    tokens = nltk.word_tokenize(sentence)
    output = list(trigrams(tokens))
    trigram_list.extend(output)

trigram_list[0:5]

In [ ]:
trigram_list[0:20]

In [ ]:
df = pl.DataFrame(
    trigram_list,
    schema=["word1", "word2", "word3"],
    orient="row",
)

df.write_parquet("trigrams.parquet")

In [4]:
counter = defaultdict(Counter)
for tokens in sentences:
    for i in range(len(tokens) - 2):
        context = (tokens[i], tokens[i + 1])   # previous 2 words
        word = tokens[i + 2]                   # next word
        counter[context][word] += 1

print("Number of contexts:", len(counter))

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x104075940>>
Traceback (most recent call last):
  File "/opt/anaconda3/envs/dsde-ice/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


Number of contexts: 6296956


In [5]:
def predict(context, first_letter):
    """
    context: tuple of two words
    first_letter: first character of target word
    """

    if context not in counter:
        return ' '

    for word, count in counter[context].most_common():
        if word.startswith(first_letter):
            return word

    return ' '

In [6]:
correct = 0
total = 0

with open("dev_set.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)

    for row in reader:
        context_words = row["context"].split()

        # Last two words form the trigram context
        trigram_context = tuple(context_words[-2:])

        first_letter = row["first letter"]
        answer = row["answer"]

        prediction = predict(trigram_context, first_letter)

        if prediction == answer:
            correct += 1

        total += 1

print("Accuracy:", correct / total)

Accuracy: 0.546997099920907
